In [25]:
import torch
import selfies as sf
from rdkit import Chem
from model_architecture import Transformer,predict 
# model_architecture.py에서 트랜스포머 모델 구조 및 예측 함수 불러오기
from rdkit.Chem import Draw
import pubchempy as pcp
from rdkit.Chem import inchi
from rdkit import Chem
from rdkit.Chem import Crippen, QED
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# cuda 있으면 cuda 쓰고 없으면 cpu 쓰기

# 체크포인트 로드
checkpoint = torch.load('D:\minimax\molecule_generation\\transformer_parameter\model_checkpoint.pt', map_location=device, weights_only=False)
token2id = checkpoint['token2id']
id2token = checkpoint['id2token']

config = checkpoint['config'].copy()
config['max_len'] = checkpoint['max_len']

# 모델 생성
model = Transformer(**config) # 모델 파라미터 정보

# 학습된 파라미터 로드
model.load_state_dict(checkpoint['model_state_dict'], strict=False)

c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


<All keys matched successfully>

In [27]:
import pubchempy as pcp
from rdkit.Chem import inchi

def find_molecule_exists(mol): # 새로 만들어진 물질이 기존에 존재하는 것인지 아닌지 확인
	inchikey = inchi.MolToInchiKey(mol)
	results = pcp.get_compounds(inchikey, "inchikey") # inchikey로 검색해서 결과가 존재하면 기존에 분자가 이미 있는 것
	if not results:
		return None

#### 리핀스키 5규칙 
- 우리가 먹는 약에는 공통적인 특성이 존재
	- (1)분자량은 500 달톤 이하 : Descriptors.MolWt(mol)
	- (2)logP <5 : Crippen.MolLogP(mol)
	- (3)수소 결합 주개가 5개 이하 
	- (4)수소 결합 받개가 10개 이하

- 이건 걍 내가 넣고 싶은 거
	- QED 계산 : 화합물이 약처럼 될 가능성을 수치화(0~1까지)
	- qed = QED.qed(mol)

In [20]:
def isit_available_medicine(mol):
	# logp 계산 : 분자가 수용성인지, 지용성인지
	logp = Crippen.MolLogP(mol)

	# QED 계산 : 화합물이 약처럼 될 가능성을 수치화(0~1까지)
	qed = QED.qed(mol)

	molecule_weight = Descriptors.MolWt(mol) # 500 이하

	hbd = Lipinski.NumHDonors(mol) # 수소 결합 주개(5개 이하)
	hba = Lipinski.NumHAcceptors(mol) # 수소 결합 받개(10개 이하)
	
	return [molecule_weight, logp, qed, hbd, hba]

In [21]:
lst = []

new_scaffold = 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'
cano_scaffold = Chem.MolToSmiles(Chem.MolFromSmiles(new_scaffold), canonical=True)
sca_list = list(sf.split_selfies(sf.encoder(new_scaffold)))
token_sca_list = [token2id[i] for i in sca_list]

res = [token2id['[SOS]']] + token_sca_list + [token2id['[EOS]']] + [token2id['[PAD]']]*(checkpoint['max_len']-len(token_sca_list))

# Here we test some examples to observe how the model predicts
example = torch.tensor([res], dtype=torch.long, device=device)

def find_key_by_value(d, value):
    return [k for k, v in d.items() if v == value]

for i in range(50):
	result = predict(model, example) # EX) [57, 55, 43, 39, 66, 45, 50, 69, 37, 64]
	sf_string = ''.join([find_key_by_value(token2id, i)[0] for i in result[1:-1]])
	# EX) [C-1][#N+1][N-1][#C-1][P][=Branch1]
	res_to_smiles,attr = sf.decoder(sf_string,attribute=True) # sf_string을 smiles로 바꾸기
	mol = Chem.MolFromSmiles(res_to_smiles) 
	mol_img = Draw.MolToImage(mol) # 분자 이미지
	if find_molecule_exists(mol) is None: # 만약 분자가 기존에 없는 것이라면
		medicine_standard = isit_available_medicine(mol) # 약이 될 수 있는지 관련 지표를 구해서 
		lst.append([f'new molecule{i}',res_to_smiles,mol_img] + medicine_standard) # DB에 저장

In [22]:
import pandas as pd 

molecule_generation = pd.DataFrame(lst,columns=['molecule_name','new_smiles','mol_image','molecule_weight', 'logP', 'QED', 'hbd', 'hba'])

In [23]:
molecule_generation

,molecule_name,new_smiles,mol_image,molecule_weight,logP,QED,hbd,hba
0,new molecule0,N#[N+1][N-1]P[N-1][CH1-1][P+1]#[C-1],<PIL.PngImagePlugin.PngImageFile image mode=RG...,144.014,2.62146,0.197229,0,1
1,new molecule1,[C-1]#[N+1][N-1][N+1][NH0][N-1]P[NH0][CH1-1]P=...,<PIL.PngImagePlugin.PngImageFile image mode=RG...,265.998,1.80258,0.270069,0,1
2,new molecule2,[C-1]#[N+1][N-1]P[N-1]C=P[N-1],<PIL.PngImagePlugin.PngImageFile image mode=RG...,144.014,2.53549,0.252679,0,0
3,new molecule3,[C-1]#[N+1][N-1]C[NH0]=P=P[N-1][N+1]#[C-1],<PIL.PngImagePlugin.PngImageFile image mode=RG...,171.040,2.99818,0.270976,0,1
4,new molecule4,[C-1]#[N+1][N-1]P[N-1],<PIL.PngImagePlugin.PngImageFile image mode=RG...,86.014,1.54089,0.263615,0,0
5,new molecule5,ClPC=[N+1]=[C-1][O-1],<PIL.PngImagePlugin.PngImageFile image mode=RG...,122.471,-0.82020,0.147218,0,1
6,new molecule6,[C-1]#[N+1][N-1]P[N-1][CH1-1][N-1][N+1]#[C-1],<PIL.PngImagePlugin.PngImageFile image mode=RG...,139.058,2.40387,0.248309,0,0
7,new molecule7,COC=C(SN=P(=O)Cl)C(C=NNC=CCl)C=O,<PIL.PngImagePlugin.PngImageFile image mode=RG...,330.133,2.31750,0.184303,1,7
8,new molecule8,[C-1]#[N+1][N-1][N+1][N-1]C=PN[N-1][N+1]#[C-1],<PIL.PngImagePlugin.PngImageFile image mode=RG...,167.072,1.33598,0.284862,1,1
9,new molecule9,FP([N-1])=P=P=P=P=PP[P+1]=P[P+1]1=P[P+1][P+1]=...,<PIL.PngImagePlugin.PngImageFile image mode=RG...,648.661,14.46679,0.111778,0,1


In [51]:
molecule_generation.to_csv('transformer_predict_result.csv',index=False)